In [0]:
# MAGIC %md
# MAGIC ### Step 1: Initialize Workflow Parameters & Catalog
# MAGIC Using dbutils.widgets allows Databricks Workflows to inject environment variables dynamically.

# 1. Define Databricks Widgets (with fallback defaults for manual testing)
dbutils.widgets.text("catalog_name", "smart-factory-catalog", "Unity Catalog Name")
dbutils.widgets.text("schema_name", "bronze", "Schema Name")
dbutils.widgets.text("base_s3_path", "s3://smart-factory-analytics-bucket", "Base S3 Path")

# 2. Retrieve values from the Workflow execution
catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
base_s3_path = dbutils.widgets.get("base_s3_path")

# 3. Dynamic Checkpoint and Schema drift locations
checkpoint_base = f"{base_s3_path}/unity-catalog/checkpoints/{schema_name}/"
schema_base = f"{base_s3_path}/unity-catalog/schemas/{schema_name}/"

# 4. Ensure database context (safely backticking identifiers to handle hyphens)
spark.sql(f"USE CATALOG `{catalog_name}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{schema_name}`")

In [0]:
# MAGIC %md
# MAGIC ### Step 2: Define the Auto Loader Function
# MAGIC Utilizing `cloudFiles` for efficient, idempotent ingestion.

from pyspark.sql.functions import current_timestamp, col

def ingest_to_bronze_idempotent(stream_name: str, source_path: str):
    """
    Incrementally and idempotently ingests raw JSON files into Bronze Delta tables.
    """
    # Safely format table name with backticks around the catalog and schema
    table_name = f"`{catalog_name}`.`{schema_name}`.{stream_name}_telemetry"
    checkpoint_path = f"{checkpoint_base}{stream_name}/"
    schema_path = f"{schema_base}{stream_name}/"
    
    print(f"Initializing idempotent stream for: {table_name}")
    
    # READ: Auto Loader Configuration
    raw_stream = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.useNotifications", "false") # Using Directory Listing
        .load(source_path)
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
    )
    
    # WRITE: Delta Table Configuration
    return (raw_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .option("default.tblproperties.delta.enableChangeDataFeed", "true")
        .trigger(availableNow=True) 
        .toTable(table_name)
    )

In [0]:
# MAGIC %md
# MAGIC ### Step 3: Execute Ingestion with Error Handling
# MAGIC Processing streams concurrently and capturing workflow state.

print("Starting Bronze Ingestion Pipeline...")

try:
    # Start the streams (Note: Verify if your Firehose drops into /temp/ or /temperature/ and adjust accordingly)
    cnc_query = ingest_to_bronze_idempotent(
        stream_name="cnc", 
        source_path=f"{base_s3_path}/cnc/"
    )
    
    conveyor_query = ingest_to_bronze_idempotent(
        stream_name="conveyor", 
        source_path=f"{base_s3_path}/conveyor/" # Corrected typo from conveyer
    )
    
    temperature_query = ingest_to_bronze_idempotent(
        stream_name="temperature", 
        source_path=f"{base_s3_path}/temp/"
    )
    
    # Block the notebook from finishing until all streams have processed the current batch
    cnc_query.awaitTermination()
    conveyor_query.awaitTermination()
    temperature_query.awaitTermination()
    
    print("✅ Bronze ingestion completed successfully. All new files processed.")

except Exception as e:
    # Explicitly catch and raise the error so the Databricks Workflow marks the run as FAILED
    print(f"❌ CRITICAL PIPELINE FAILURE: {str(e)}")
    raise e

In [0]:
%sql
SELECT * FROM `smart-factory-catalog`.bronze.cnc_telemetry
ORDER BY ingestion_timestamp DESC
LIMIT 10;